# PDF to RDA DMP JSON

One prompt: the text of a DMP PDF plus the complete **maDMP 1.2** schema, with
strict instructions to follow the schema. Run first with `llama3.1:8b`, then the
same prompt with `gemma4:e4b` and `llama3.3:70b`.

| Step | What happens |
|---|---|
| 1 | Read sample 14 with pdfplumber |
| 2 | Load the schema — unchanged |
| 3 | Prompt → `llama3.1:8b` |
| 4 | Same prompt → `gemma4:e4b` |
| 5 | Same prompt → `llama3.3:70b` |
| 6 | Save all three |


In [ ]:
import json
from pathlib import Path

if Path.cwd().name == "notebooks":
    import os
    os.chdir(Path.cwd().parent)

from dmpbridge.extractors import get_extractor
from dmpbridge.models.ollama import OllamaModel

PDF     = Path("data/input/pdfs/sample14.pdf")
SCHEMA  = Path("data/output/rda/maDMP-schema-1.2.json")
HOST    = "http://localhost:11434"
MODEL_1 = "llama3.1:8b"
MODEL_2 = "gemma4:e4b"
MODEL_3 = "llama3.3:70b"
OUT_DIR = Path("data/output/rda")


## Step 1 — Read the PDF


In [ ]:
dmp_text = get_extractor("pdfplumber").extract(PDF)[0]["text"]

print(f"{len(dmp_text):,} characters\n")
print(dmp_text[:500])


## Step 2 — Load the schema

The schema is used exactly as published — nothing removed, nothing changed. Its
`$ref` pointers are resolved to the definitions they point to, so the model sees
the schema in one piece instead of copying `"$ref"` into its answer.


In [ ]:
schema = json.loads(SCHEMA.read_text(encoding="utf-8"))
defs = schema["$defs"]


def resolve_refs(node):
    """Replace every $ref with the definition it points at. Content unchanged."""
    if isinstance(node, dict):
        if "$ref" in node:
            return resolve_refs(defs[node["$ref"].split("/")[-1]])
        return {k: resolve_refs(v) for k, v in node.items() if k != "$defs"}
    if isinstance(node, list):
        return [resolve_refs(v) for v in node]
    return node


schema_full = resolve_refs(schema)
schema_text = json.dumps(schema_full, separators=(",", ":"))

print(f"{len(schema_text):,} characters of schema, {len(defs)} definitions resolved")


## Step 3 — Prompt → llama3.1:8b

The schema is given to the model twice: as text in the prompt, and as Ollama's
`format`, which constrains the JSON it generates to that structure. Without the
constraint, `llama3.1:8b` fell into a loop on this prompt and never stopped;
`num_predict` is a cap in case it ever does again.


In [ ]:
import subprocess
import time

SYSTEM = """You convert Data Management Plans into RDA maDMP JSON.

Strict rules:
1. Use only the field names defined in the schema. Never add a key that is not in the schema.
2. Put every field exactly where the schema places it. The whole document is one top-level "dmp" object.
3. Where the schema lists allowed values, use one of them, spelled exactly as in the schema.
4. Take every value from the Data Management Plan text. Never copy example values from the schema.
5. Output only the JSON object. No explanation, no markdown."""

PROMPT = f"""Here is the RDA maDMP JSON schema (version 1.2). Follow it strictly:

{schema_text}

Here is the text of a Data Management Plan:

{dmp_text}

Generate one JSON object that strictly follows the schema above, filling in
whatever information the Data Management Plan text contains. Output only the JSON."""


def run(model):
    """Send the prompt to one model; the schema is also the output constraint.
    Prints how long it took and how many tokens went in and came out."""
    for other in (MODEL_1, MODEL_2, MODEL_3):
        if other != model:
            subprocess.run(["ollama", "stop", other], check=False)   # one model in VRAM at a time

    llm = OllamaModel(model=model, host=HOST, num_ctx=32768, num_predict=8000)
    t0 = time.perf_counter()
    result = json.loads(llm.complete(SYSTEM, PROMPT, schema=schema_full))
    elapsed = time.perf_counter() - t0

    s = llm.last_call
    print(f"{model}: {elapsed:.0f} s total, of which model load {s['load_duration'] / 1e9:.0f} s")
    print(f"tokens sent to the model: {s['prompt_eval_count']:,}   tokens generated: {s['eval_count']:,}")
    print()
    return result


result_llama = run(MODEL_1)
print(json.dumps(result_llama, indent=2, ensure_ascii=False))


## Step 4 — Same prompt → gemma4:e4b


In [ ]:
result_gemma = run(MODEL_2)
print(json.dumps(result_gemma, indent=2, ensure_ascii=False))


## Step 5 — Same prompt → llama3.3:70b

The 70B needs about 42 GB of VRAM; `run()` unloads the other models first.
This call takes a few minutes, most of it loading the model.


In [ ]:
result_llama33 = run(MODEL_3)
print(json.dumps(result_llama33, indent=2, ensure_ascii=False))


## Step 6 — Save all three


In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
for model, result in ((MODEL_1, result_llama), (MODEL_2, result_gemma),
                      (MODEL_3, result_llama33)):
    out = OUT_DIR / f"{PDF.stem}.rda.{model.replace(':', '-')}.json"
    out.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"{model:14} -> {out}")
